In [12]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as imbpipeline
from joblib import dump
import warnings
warnings.filterwarnings("ignore")

SEED = 42

# Paths
processed_data_dir = os.path.join('..', 'data', 'processed')
results_output_path = os.path.join('..', 'results', 'model_evaluation_results.csv')
best_models_dir = os.path.join('..', 'models', 'best_pipelines')
os.makedirs(best_models_dir, exist_ok=True)

# Model definitions
models = {
    "LightGBM": LGBMClassifier(
        objective='binary', metric='auc', boosting_type='gbdt',
        num_leaves=31, learning_rate=0.05, n_estimators=100, random_state=SEED
    ),
    "CatBoost": CatBoostClassifier(
        iterations=500, learning_rate=0.03, depth=6,
        loss_function='Logloss', eval_metric='AUC',
        auto_class_weights='Balanced', verbose=False, random_state=SEED
    ),
    "LogisticRegression": LogisticRegression(
        class_weight='balanced', max_iter=5000, solver='saga', random_state=SEED
    ),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(64, 32), activation='relu', solver='adam',
        max_iter=1000, early_stopping=True, validation_fraction=0.2, random_state=SEED
    )
}

# Load processed files
data_files = [f for f in os.listdir(processed_data_dir) if f.endswith('.pkl')]
global_results = []

# Cross-validator
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Main loop (skip test_final during cross-validation)
for file_name in data_files:
    if "test_final" in file_name:
        print(f"Skipping {file_name} from cross-validation")
        continue

    print(f"Evaluating models for file: {file_name}")
    file_path = os.path.join(processed_data_dir, file_name)
    Data = pd.read_pickle(file_path)

    X = Data.drop(columns=['target'], errors='ignore')  
    y = Data['target']  

    # Skip if not enough samples
    if len(X) < skf.n_splits or any(y.value_counts() < skf.n_splits):
        print(f"Skipping {file_name} due to insufficient samples.")
        continue

    for model_name, model in models.items():
        print(f"Training {model_name} on {file_name}")
        cv_results = []

        for train_idx, test_idx in skf.split(X, y):
            X_train, X_val = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[test_idx]

            # Check for NaNs before applying SMOTE
            if X_train.isnull().any().any() or X_val.isnull().any().any():
                print(f"⚠️ Skipping fold due to NaN values in {file_name}")
                continue

            pipeline = imbpipeline([
                ('scaler', StandardScaler()),
                ('smote', SMOTE(random_state=SEED)),
                ('clf', model)
            ])

            try:
                pipeline.fit(X_train, y_train)
                y_pred_proba = pipeline.predict_proba(X_val)[:, 1]
                y_pred = (y_pred_proba > 0.5).astype(int)

                roc_auc = roc_auc_score(y_val, y_pred_proba)
            except Exception as e:
                print(f"Error evaluating {model_name}: {e}")
                continue

            cv_results.append({
                'ROC AUC': roc_auc,
                'Accuracy': accuracy_score(y_val, y_pred),
                'Precision': precision_score(y_val, y_pred, zero_division=0),
                'Recall': recall_score(y_val, y_pred, zero_division=0),
                'F1': f1_score(y_val, y_pred, zero_division=0),
                
            })

        if not cv_results:
            print(f"No valid folds for {file_name}, skipping saving results...")
            continue

        avg_results = {
            'Processed_File': file_name,
            'Model': model_name,
            'ROC AUC': np.mean([r['ROC AUC'] for r in cv_results]),
            'Accuracy': np.mean([r['Accuracy'] for r in cv_results]),
            'Precision': np.mean([r['Precision'] for r in cv_results]),
            'Recall': np.mean([r['Recall'] for r in cv_results]),
            'F1': np.mean([r['F1'] for r in cv_results])
        }
        global_results.append(avg_results)

# Save evaluation results
results_df = pd.DataFrame(global_results)
results_df.to_csv(results_output_path, index=False)
print("\n✅ Evaluation completed. Results saved.")

# Optional: Evaluate best model on test_final.pkl
try:
    test_file = "Data_test_final.pkl"
    test_path = os.path.join(processed_data_dir, test_file)
    test_data = pd.read_pickle(test_path)

    X_test = test_data.drop(columns=['target'])
    y_test = test_data['target']

    best_pipeline = imbpipeline([
        ('scaler', StandardScaler()),
        ('clf', LGBMClassifier(random_state=SEED))
    ])

    best_pipeline.fit(X_test, y_test)  # No SMOTE here
    y_pred_proba = best_pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba > 0.5).astype(int)

    test_result = {
        'Processed_File': test_file,
        'Model': 'Final_LightGBM_on_Test',
        'ROC AUC': roc_auc_score(y_test, y_pred_proba),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'Fold': fold_idx
    }

    results_df = pd.concat([results_df, pd.DataFrame([test_result])], ignore_index=True)
    results_df.to_csv(results_output_path, index=False)
    dump(best_pipeline, os.path.join(best_models_dir, 'final_lightgbm_pipeline.joblib'))
    print(f"\n✅ Final model evaluated on {test_file} and saved.")

except Exception as e:
    print(f"❌ Could not evaluate final model on test set: {e}")

print("\n✅ Full training process finished.")

Evaluating models for file: Data_drop_na.pkl
Training LightGBM on Data_drop_na.pkl
[LightGBM] [Info] Number of positive: 1139, number of negative: 1139
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002726 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16945
[LightGBM] [Info] Number of data points in the train set: 2278, number of used features: 73
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Number of positive: 1139, number of negative: 1139
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003827 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16477
[LightGBM] [Info] Number of data points in the train set: 2278, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Number of positive: 1139,

In [13]:
# Load test_final without resampling
test_final = pd.read_pickle(os.path.join(processed_data_dir, "Data_test_final.pkl"))
X_test = test_final.drop(columns=['target'])
y_test = test_final['target']

# Train best model on full train set
best_pipeline = imbpipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=SEED)),
    ('clf', LGBMClassifier(random_state=SEED))
])

best_pipeline.fit(X, y)

# Evaluate on clean test set (no SMOTE applied!)
y_pred_proba = best_pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

print("Final Test ROC AUC:", roc_auc_score(y_test, y_pred_proba))
print("Final Test F1 Score:", f1_score(y_test, y_pred))

[LightGBM] [Info] Number of positive: 30286, number of negative: 30286
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.033115 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 19140
[LightGBM] [Info] Number of data points in the train set: 60572, number of used features: 79
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Final Test ROC AUC: 0.5612917478548209
Final Test F1 Score: 0.0536231884057971
